# Dataset visualizer

#### Imports and data loading

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown, HTML, clear_output
import ipywidgets as widgets

# Path configuration
DATASET_PATH = "data/curated_dataset.jsonl"

def load_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return pd.DataFrame(data)

# Master DataFrame Initialization
df = load_data(DATASET_PATH)

print("Dataset loaded successfully!")
print(f"Total interactions: {len(df)}")
print(f"Total base scenarios: {df['group'].nunique()}")

Dataset loaded successfully!
Total interactions: 630
Total base scenarios: 30


#### Data visualization

In [27]:
# Extract unique biases dynamically
all_biases = sorted([b for b in df['bias_type'].dropna().unique() if b != "control"])

# Explicit component declaration for Casillas (Checkboxes)
scenario_slider = widgets.IntSlider(
    min=0, max=df['group'].max(), value=0, 
    description="Group ID:", continuous_update=False
)

bias_boxes = widgets.SelectMultiple(
    options=all_biases,
    value=all_biases,
    description="Biases:",
    disabled=False,
    rows=10,
    layout=widgets.Layout(width='20%')
)

# Label helper text to display right above the selector
helper_text = widgets.HTML(
    value="<i>Hold <b>Ctrl</b> (or <b>Cmd</b> on Mac) to select multiple biases seamlessly.</i>"
)

# Main Explorer Logic
def explorer_cell(TARGET_GROUP_ID, SELECTED_BIASES):
    clear_output(wait=True)
    
    # Filter current scenario group
    group_df = df[df['group'] == TARGET_GROUP_ID]

    # Extract Control Row (matches NaN or "control")
    control_mask = group_df['bias_type'].isna() | (group_df['bias_type'] == "control")
    if not group_df[control_mask].empty:
        control_row = group_df[control_mask].iloc[0]
        original_text = control_row['dilemma_situation']
        basic_sit_label = control_row['basic_situation']
    else:
        original_text = "No control prompt found."
        basic_sit_label = "N/A"

    # Filter Induced items based on selected biases
    induced_items = group_df[group_df['bias_type'].isin(SELECTED_BIASES)]

    # --- RENDER DISPLAY WITH HTML & MARKDOWN ---
    display(Markdown("<br><br>"))
    display(Markdown(f"# Sycophancy Multi-Bias Explorer — Scenario {TARGET_GROUP_ID}"))
    display(Markdown("<br>"))
    display(Markdown(f"**Core Domain Focus:** *{basic_sit_label}*"))
    display(Markdown("<br><br>"))

    # Control Box using hardcoded HTML/CSS for absolute contrast (White text over Dark Blue info box)
    control_html = f"""
    <div style="background-color: #1e3a8a; padding: 15px; border-left: 6px solid #3b82f6; border-radius: 4px; margin-bottom: 10px;">
        <h3 style="color: #ffffff; margin-top: 0; font-weight: bold; font-size: 1.1em;">📝 CONTROL PROMPT (Original baseline)</h3>
        <p style="color: #f3f4f6; margin-bottom: 0; font-style: italic;"><strong>Situation:</strong> {original_text}</p>
    </div>
    """
    display(HTML(control_html))
    display(Markdown("---"))

    # Display Induced Variations
    display(Markdown(f"### Induced Variations ({len(induced_items)})"))
    display(Markdown("<br>"))

    if induced_items.empty:
        display(Markdown("`(!) No variations match the selected filters.`"))

    for _, item in induced_items.iterrows():
        induced_text = item['dilemma_situation']
        
        # Header for each variation
        display(Markdown(f"#### 🔹 ID {item['idx']} | Bias: `{item['bias_type'].upper()}`"))        
        
        # Smart Diff Logic
        if induced_text.startswith(original_text):
            bias_injection = induced_text[len(original_text):].strip()
            display(Markdown("**[Base Text matches Control]**"))
            
            # Bias Injection using hardcoded HTML/CSS (White text over Emerald Green box)
            injection_html = f"""
            <div style="background-color: #065f46; padding: 12px; border-left: 6px solid #10b981; border-radius: 4px; margin-top: 5px; margin-bottom: 15px;">
                <span style="color: #a7f3d0; font-weight: bold; font-size: 0.9em;">INJECTED BIAS PRESSURE:</span>
                <p style="color: #ffffff; font-family: monospace; font-size: 1.05em; margin-top: 5px; margin-bottom: 0; white-space: pre-wrap;">+ {bias_injection}</p>
            </div>
            """
            display(HTML(injection_html))
        else:
            # Fallback if text structural mutation is different
            display(Markdown(f"**Full Custom Prompt:**\n{induced_text}"))
        
        display(Markdown("<br>"))

# Bind widgets to the layout engine symmetrically
out = widgets.interactive_output(
    explorer_cell, 
    {'TARGET_GROUP_ID': scenario_slider, 'SELECTED_BIASES': bias_boxes}
)

# Display controls stacked vertically over the content output
display(widgets.VBox([scenario_slider, helper_text, bias_boxes, out]))